# nn-module-subclass composite — cx23: custom ReLU as a parameterless nn.Module: max(x, 0)

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `nn-module-subclass`, `relu-elementwise-max`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import torch.nn as nn
import torch.nn.functional as F

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "nn-module-subclass"
DD_ATOM_IDS = ["nn-module-subclass", "relu-elementwise-max"]
DD_SUBTOPICS = ["PyTorch: nn.Module subclassing", "CNN: ReLU as elementwise max"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## How these two atoms compose

Even a parameterless layer is worth wrapping as an `nn.Module` — it slots cleanly into `nn.Sequential`, shows up in the repr tree, and the `.train()/.eval()` toggle reaches it.
- **nn-module-subclass** — the usual scaffold, but with NOTHING registered in `__init__` beyond `super().__init__()` (no params, no buffers, no children).
- **relu-elementwise-max** — the math: `relu(x) = max(x, 0)`. Implement as `t.maximum(x, t.zeros_like(x))` (explicit elementwise max). Avoid `t.relu` / `F.relu` — those would dodge the atom.

**Why max(x, 0) and not `x * (x > 0)`?** The mask form loses the gradient signal at zero in a different way; `t.maximum` is the canonical PyTorch implementation pattern and matches what ARENA expects. (At exactly `x == 0` both forms have subgradient ambiguity, but `maximum` consistently picks the right branch.)

**Anatomy.**
1. `super().__init__()` — that's the entire `__init__`.
2. `forward(self, x): return t.maximum(x, t.zeros_like(x))`.
3. `.parameters()` returns an EMPTY iterator — no learnables. The test verifies this.

### Composite Exercise — custom ReLU as a parameterless nn.Module: max(x, 0)

**Atoms exercised together**: `nn-module-subclass`, `relu-elementwise-max`

Define a class `MyReLU(nn.Module)` and a builder `cx23_build_relu()`.

`MyReLU.__init__` calls `super().__init__()` and does NOTHING else — no parameters, no buffers, no children.

`MyReLU.forward(self, x)` must return `t.maximum(x, t.zeros_like(x))` — the elementwise max against zero.

**Forbidden shortcuts** (these dodge the atom):
- `t.relu(x)` / `F.relu(x)` / `nn.functional.relu(x)` — would not exercise the elementwise max construction.
- `x.clamp(min=0)` — also a shortcut. The drill is specifically about the `t.maximum` pattern.

(The test cannot easily detect which built-in you used since results match — but on the honor system, write `t.maximum(...)`.)

In [ ]:
# Fill in the function below, then run this cell. The test asserts the composition is correct.

class MyReLU(nn.Module):
    def __init__(self):
        raise NotImplementedError

    def forward(self, x):
        raise NotImplementedError


def cx23_build_relu() -> 'MyReLU':
    raise NotImplementedError

def _test_cx23():
    # Case A: parameterless — no learnables, no buffers, no children.
    m = cx23_build_relu()
    assert isinstance(m, nn.Module)
    assert list(m.parameters()) == [], 'ReLU has no parameters — got some'
    assert list(m.buffers()) == [], 'ReLU has no buffers — got some'
    assert list(m.children()) == [], 'ReLU has no submodules — got some'

    # Case B: numerics on a hand-built case covering negative, zero, positive.
    x = t.tensor([-2.0, -0.1, 0.0, 0.1, 3.0])
    y = m(x)
    assert tuple(y.shape) == (5,)
    assert t.equal(y, t.tensor([0.0, 0.0, 0.0, 0.1, 3.0]))

    # Case C: matches the reference t.relu / F.relu (math agreement, even though the
    # implementation should use t.maximum internally).
    t.manual_seed(0)
    x = t.randn(4, 5)
    y = m(x)
    assert t.equal(y, t.relu(x))
    assert t.equal(y, F.relu(x))

    # Case D: shape pass-through — 4-D input goes in, same shape comes out.
    x = t.randn(2, 3, 4, 5)
    y = m(x)
    assert tuple(y.shape) == (2, 3, 4, 5)
    assert (y >= 0).all().item(), 'ReLU output must be non-negative everywhere'

    # Case E: gradient through the positive branch is 1, through the negative branch is 0.
    x = t.tensor([-1.0, 2.0, -3.0, 4.0], requires_grad=True)
    y = m(x)
    y.sum().backward()
    assert t.equal(x.grad, t.tensor([0.0, 1.0, 0.0, 1.0])), f'wrong grads: {x.grad}'

    # Case F: composes into nn.Sequential cleanly — slots in like nn.ReLU().
    stack = nn.Sequential(nn.Linear(4, 3), cx23_build_relu())
    x = t.randn(2, 4)
    y = stack(x)
    assert tuple(y.shape) == (2, 3)
    assert (y >= 0).all().item()
    _dd_passed.add('cx23')

_test_cx23()

<details><summary>Show solution — cx23</summary>

```python
class MyReLU(nn.Module):
    def __init__(self):
        # Atom A (nn-module-subclass): super().__init__() is the WHOLE __init__ for
        # parameterless layers — no params, no buffers, no children to register.
        super().__init__()

    def forward(self, x):
        # Atom B (relu-elementwise-max): the canonical elementwise max(x, 0) form.
        # t.maximum broadcasts the zero tensor against x and picks the larger at each cell.
        return t.maximum(x, t.zeros_like(x))


def cx23_build_relu() -> 'MyReLU':
    return MyReLU()
```

Wrapping ReLU as a Module (vs leaving it as a free function) lets it participate in the module tree — appears in `repr(model)`, slots into `nn.Sequential`, gets toggled by `.train()/.eval()` (irrelevant for ReLU but consistent), and shows up in `state_dict()` (empty for parameterless layers, but consistent).
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx23'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx23',
        'subtopics': ["PyTorch: nn.Module subclassing", "CNN: ReLU as elementwise max"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()